In [1]:
import os
for f in os.listdir('/kaggle/input/'):
    print(f)
    path = f'/kaggle/input/{f}'
    if os.path.isdir(path):
        for ff in os.listdir(path):
            print(f'  {ff}')

datasets
  tlarkdus


In [2]:
import os
for f in os.listdir('/kaggle/input/datasets/tlarkdus/dataset'):
    print(f)

TR_preprocessed.csv
VL_preprocessed.csv


# **라이브러리 + GPU 확인**

In [3]:
!pip install transformers accelerate sentence-transformers -q

import torch
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU 사용 가능: True
GPU: Tesla T4


# **데이터 로드 + 30% 샘플링:**

In [4]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# Kaggle 데이터 경로
TR_PATH = '/kaggle/input/datasets/tlarkdus/dataset/TR_preprocessed.csv'
VL_PATH = '/kaggle/input/datasets/tlarkdus/dataset/VL_preprocessed.csv'
SAVE_DIR = '/kaggle/working/models'
os.makedirs(SAVE_DIR, exist_ok=True)

tr_df = pd.read_csv(TR_PATH)
vl_df = pd.read_csv(VL_PATH)
print(f"원본 Training: {len(tr_df):,}건 / Validation: {len(vl_df):,}건")

# Part별 분리
train_p1 = tr_df[tr_df['partNum'] == 'P1'].reset_index(drop=True)
train_p2 = tr_df[tr_df['partNum'] == 'P2'].reset_index(drop=True)
vl_p1 = vl_df[vl_df['partNum'] == 'P1'].reset_index(drop=True)
vl_p2 = vl_df[vl_df['partNum'] == 'P2'].reset_index(drop=True)

# 30% 층화 샘플링 — Part별 87,000건
tr_p1, _ = train_test_split(train_p1, train_size=87000, stratify=train_p1['clickbaitClass'], random_state=42)
tr_p2, _ = train_test_split(train_p2, train_size=87000, stratify=train_p2['clickbaitClass'], random_state=42)
tr_p1 = tr_p1.reset_index(drop=True)
tr_p2 = tr_p2.reset_index(drop=True)

print(f"\n✅ 30% 샘플링 완료!")
print(f"[Part1] Training: {len(tr_p1):,}건 / Validation: {len(vl_p1):,}건")
print(f"[Part2] Training: {len(tr_p2):,}건 / Validation: {len(vl_p2):,}건")

원본 Training: 586,741건 / Validation: 73,344건



✅ 30% 샘플링 완료!
[Part1] Training: 87,000건 / Validation: 36,434건
[Part2] Training: 87,000건 / Validation: 36,910건


# **입력 텍스트 구성**

In [5]:
tr_p1['input_text'] = tr_p1['fakeTitle'].fillna('') + ' [SEP] ' + tr_p1['newsContent'].fillna('')
vl_p1['input_text'] = vl_p1['fakeTitle'].fillna('') + ' [SEP] ' + vl_p1['newsContent'].fillna('')
tr_p2['input_text'] = tr_p2['newsTitle'].fillna('') + ' [SEP] ' + tr_p2['newsContent'].fillna('')
vl_p2['input_text'] = vl_p2['newsTitle'].fillna('') + ' [SEP] ' + vl_p2['newsContent'].fillna('')
print("✅ 입력 텍스트 구성 완료!")

✅ 입력 텍스트 구성 완료!


# **Dataset + 학습/평가 함수**

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import numpy as np

class ClickbaitDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="학습 중"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="평가 중"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')

    print(f"\n📊 평가 결과:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  F1-score:  {f1:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"\n{classification_report(all_labels, all_preds, target_names=['낚시성(0)', '정상(1)'])}")
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

print("✅ 함수 정의 완료!")

✅ 함수 정의 완료!


# **실험 실행 함수**

In [7]:
def run_experiment(exp_name, model_name, tr_texts, tr_labels, vl_texts, vl_labels,
                   num_epochs=3, batch_size=16, lr=1e-5):
    print(f"\n{'=' * 60}")
    print(f"🚀 {exp_name} 시작")
    print(f"  모델: {model_name}")
    print(f"  Training: {len(tr_texts):,}건 / Validation: {len(vl_texts):,}건")
    print(f"{'=' * 60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    tr_dataset = ClickbaitDataset(tr_texts, tr_labels, tokenizer)
    vl_dataset = ClickbaitDataset(vl_texts, vl_labels, tokenizer)
    tr_loader = DataLoader(tr_dataset, batch_size=batch_size, shuffle=True)
    vl_loader = DataLoader(vl_dataset, batch_size=batch_size, shuffle=False)

    optimizer = AdamW(model.parameters(), lr=lr)

    best_f1 = 0
    best_results = None

    for epoch in range(num_epochs):
        print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")
        avg_loss = train_epoch(model, tr_loader, optimizer, device)
        print(f"평균 Loss: {avg_loss:.4f}")

        print(f"\n[Validation 평가]")
        results = evaluate(model, vl_loader, device)

        if results['f1'] > best_f1:
            best_f1 = results['f1']
            best_results = results
            save_path = os.path.join(SAVE_DIR, f"{exp_name}_best.pt")
            torch.save(model.state_dict(), save_path)
            print(f"💾 최고 모델 저장! (F1: {best_f1:.4f})")

    print(f"\n{'=' * 60}")
    print(f"✅ {exp_name} 완료! 최고 F1: {best_f1:.4f}")
    print(f"{'=' * 60}")

    del model, tokenizer, tr_dataset, vl_dataset, tr_loader, vl_loader, optimizer
    torch.cuda.empty_cache()

    return {'experiment': exp_name, 'model': model_name, 'best_f1': best_f1, **best_results}

print("✅ 실험 실행 함수 정의 완료!")

✅ 실험 실행 함수 정의 완료!


# **모델 매핑**

In [8]:
MODELS = {
    'KoBERT': 'skt/kobert-base-v1',
    'KLUE-RoBERTa': 'klue/roberta-base',
    'KPF-BERT': 'jinmang2/kpfbert'
}
print("모델 목록:")
for name, path in MODELS.items():
    print(f"  {name}: {path}")

모델 목록:
  KoBERT: skt/kobert-base-v1
  KLUE-RoBERTa: klue/roberta-base
  KPF-BERT: jinmang2/kpfbert


In [9]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# tokenizer = AutoTokenizer.from_pretrained('skt/kobert-base-v1')
# model = AutoModelForSequenceClassification.from_pretrained('skt/kobert-base-v1', num_labels=2)
# print("✅ skt/kobert-base-v1 로드 성공!")

# **실험 셀**

In [10]:
# === 실험 1-1: Part1 + KoBERT ===
results_1_1 = run_experiment(
    exp_name='exp1_1_P1_KoBERT',
    model_name=MODELS['KoBERT'],
    tr_texts=tr_p1['input_text'].tolist(),
    tr_labels=tr_p1['clickbaitClass'].tolist(),
    vl_texts=vl_p1['input_text'].tolist(),
    vl_labels=vl_p1['clickbaitClass'].tolist()
)


🚀 exp1_1_P1_KoBERT 시작
  모델: skt/kobert-base-v1
  Training: 87,000건 / Validation: 36,434건


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

... (414 outputs left out)

학습 중:   8%|▊         | 411/5438 [11:33<2:22:32,  1.70s/it]

학습 중:   8%|▊         | 412/5438 [11:34<2:22:56,  1.71s/it]

학습 중:   8%|▊         | 413/5438 [11:36<2:23:22,  1.71s/it]

학습 중:   8%|▊         | 414/5438 [11:38<2:23:18,  1.71s/it]

In [ ]:
# === 실험 1-2: Part1 + KLUE-RoBERTa ===
results_1_2 = run_experiment(
    exp_name='exp1_2_P1_KLUE_RoBERTa',
    model_name=MODELS['KLUE-RoBERTa'],
    tr_texts=tr_p1['input_text'].tolist(),
    tr_labels=tr_p1['clickbaitClass'].tolist(),
    vl_texts=vl_p1['input_text'].tolist(),
    vl_labels=vl_p1['clickbaitClass'].tolist()
)

In [ ]:
# === 실험 1-3: Part1 + KPF-BERT ===
results_1_3 = run_experiment(
    exp_name='exp1_3_P1_KPF_BERT',
    model_name=MODELS['KPF-BERT'],
    tr_texts=tr_p1['input_text'].tolist(),
    tr_labels=tr_p1['clickbaitClass'].tolist(),
    vl_texts=vl_p1['input_text'].tolist(),
    vl_labels=vl_p1['clickbaitClass'].tolist()
)

In [ ]:
# === 실험 2-1: Part2 + KoBERT ===
results_2_1 = run_experiment(
    exp_name='exp2_1_P2_KoBERT',
    model_name=MODELS['KoBERT'],
    tr_texts=tr_p2['input_text'].tolist(),
    tr_labels=tr_p2['clickbaitClass'].tolist(),
    vl_texts=vl_p2['input_text'].tolist(),
    vl_labels=vl_p2['clickbaitClass'].tolist()
)

In [ ]:
# === 실험 2-2: Part2 + KLUE-RoBERTa ===
results_2_2 = run_experiment(
    exp_name='exp2_2_P2_KLUE_RoBERTa',
    model_name=MODELS['KLUE-RoBERTa'],
    tr_texts=tr_p2['input_text'].tolist(),
    tr_labels=tr_p2['clickbaitClass'].tolist(),
    vl_texts=vl_p2['input_text'].tolist(),
    vl_labels=vl_p2['clickbaitClass'].tolist()
)

In [ ]:
# === 실험 2-3: Part2 + KPF-BERT ===
results_2_3 = run_experiment(
    exp_name='exp2_3_P2_KPF_BERT',
    model_name=MODELS['KPF-BERT'],
    tr_texts=tr_p2['input_text'].tolist(),
    tr_labels=tr_p2['clickbaitClass'].tolist(),
    vl_texts=vl_p2['input_text'].tolist(),
    vl_labels=vl_p2['clickbaitClass'].tolist()
)